In [ ]:
!pip uninstall -y peft
!pip install --no-cache-dir "peft>=0.19"

!pip install -q \
    langgraph==1.2.11 \
    faiss-cpu \
    sentence-transformers==6.0.1 \
    datasets==3.2.0 \
    python-dotenv==1.0.1 \
    pydantic==2.10.4 \
    transformers \
    accelerate \

print("Dependencies installed.")

In [ ]:
print("testing ")

In [ ]:
!pip show peft

In [ ]:
import peft
print("peft version is: ", peft.__version__)

In [ ]:
import peft
print(peft.__version__)

In [ ]:
!pip install faiss-cpu
import faiss

In [ ]:
# config.py All magic numbers in one place.

# Local model served via HuggingFace Transformers (no API key needed)
LOCAL_MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
GROQ_MODEL       = LOCAL_MODEL_ID   # alias kept so nothing downstream breaks

EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE       = 500
CHUNK_OVERLAP    = 50
TOP_K            = 4
MAX_RETRIES      = 2

DATASET_NAME     = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
DATASET_SPLIT    = "train"

print(f"Config loaded. Local model: {LOCAL_MODEL_ID}")



In [ ]:
# No API key required
# The pipeline now runs entirely on the local Qwen2.5-7B-Instruct model.
# No GEMINI_API_KEY or GROQ_API_KEY needed.
import os
print("Running fully locally — no API key needed.")



In [ ]:
# agents/prompts/

ROUTER_SYSTEM_PROMPT = (
    'You are an intent classifier for a customer-support system.\n'
    'Return a SINGLE intent label (exact, lowercase_underscored):\n'
    '  account_access, order_management, shipping_delivery, returns_refunds,\n'
    '  billing_payment, product_information, technical_support, general_inquiry,\n'
    '  contact_human_agent, contact_customer_service\n'
    'Respond with ONLY the label — no punctuation, no explanation.'
)

def build_router_messages(query: str) -> list:
    return [
        {'role': 'system', 'content': ROUTER_SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Query: {query}'},
    ]

GENERATOR_SYSTEM_PROMPT = GENERATOR_SYSTEM_PROMPT = (
    'You are a helpful customer-support assistant.\n'
    'Answer ONLY using the provided context excerpts. Do NOT fabricate information.\n'
    'If context is insufficient, say what you cannot confirm.\n'
    'Be concise (2-5 sentences unless the topic genuinely requires more).'
)

def build_generator_messages(query: str, chunks: list) -> list:
    ctx = '\n\n'.join(f'[Excerpt {i+1}]\n{c}' for i, c in enumerate(chunks))
    user_msg = f'--- CONTEXT EXCERPTS ---\n{ctx}\n--- END ---\n\nCustomer question: {query}\n\nGrounded answer:'
    return [
        {'role': 'system', 'content': GENERATOR_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]


CRITIC_SYSTEM_PROMPT = (
    'You are a factual-grounding critic for a customer-support AI.\n'
    'Verify whether every claim in the answer is supported by the context excerpts.\n'
    'IMPORTANT: if the answer declines to answer, says it cannot confirm something, '
    'or states the information is not available/not covered by context — treat this '
    'as grounded=false, since it means the context did not actually answer the '
    'customer\'s question and the query should be escalated, not treated as resolved.\n'
    'Return ONLY this JSON object (no markdown fences, no extra keys):\n'
    '{"grounded": true/false, "unsupported_claims": ["..."], "reason": "..."}\n'
    'grounded=true ONLY if the answer both (a) makes claims fully supported by the '
    'context, AND (b) actually addresses the customer\'s question.\n'
    'unsupported_claims: verbatim phrases not backed by any excerpt (empty list if grounded=true).'
)

def build_critic_messages(answer: str, chunks: list) -> list:
    ctx = '\n\n'.join(f'[Excerpt {i+1}]\n{c}' for i, c in enumerate(chunks))
    user_msg = f'--- CONTEXT ---\n{ctx}\n--- END ---\n\n--- ANSWER ---\n{answer}\n--- END ---\n\nReturn JSON verdict.'
    return [
        {'role': 'system', 'content': CRITIC_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]




In [ ]:
!pip install -U "bitsandbytes>=0.46.1" accelerate


In [ ]:
import json, logging, re, torch
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from types import SimpleNamespace
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("rag_pipeline")


# agents/state.py shared LangGraph state schema
class SupportState(TypedDict, total=False):
    query: str
    intent: str
    retrieved_chunks: list
    draft_answer: str
    grounded: bool
    retry_count: int
    final_answer: str
    escalated: bool
    node_path: list
    unsupported_claims: list
    critic_reason: str


# agents/critic.py CriticResult structured, eval-harness-ready verdict
class CriticResult(BaseModel):
    grounded: bool           = Field(..., description="True if every claim is grounded.")
    unsupported_claims: list = Field(default_factory=list)
    reason: str              = Field(..., description="One-sentence explanation.")


# local_model/client.py────────────────────────────────────────────────
# Loads Qwen2.5-7B-Instruct in 4-bit NF4 and exposes the same
# .chat.completions.create() surface as the old GeminiClient so that
# _llm() in the agent-nodes cell is completely untouched.

class LocalModelClient:
    """HuggingFace Transformers backend with an OpenAI-compatible surface."""

    def __init__(self, model_id: str):
        print(f"Loading {model_id} in 4-bit NF4 … (~4 GB download on first run)")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        self._tok = AutoTokenizer.from_pretrained(model_id)
        self._model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self._model.eval()
        device = next(self._model.parameters()).device
        print(f"{model_id} loaded on {device}")
        # Mirror GeminiClient attribute chain so client.chat.completions.create() works
        self.chat        = self
        self.completions = self

    @torch.inference_mode()
    def create(self, model=None, messages=None,
               temperature=0.2, max_tokens=512, **kwargs):
        """
        messages : list of {role, content} dicts (system / user / assistant).
        Returns a SimpleNamespace that looks like an OpenAI ChatCompletion object.
        The `model` kwarg is accepted but ignored — the loaded model is always used.
        """
        messages = messages or []
        prompt = self._tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self._tok(prompt, return_tensors="pt").to(self._model.device)

        gen_kwargs = dict(
            max_new_tokens=max_tokens,
            pad_token_id=self._tok.eos_token_id,
            eos_token_id=self._tok.eos_token_id,
        )
        if temperature == 0.0:
            gen_kwargs["do_sample"] = False
        else:
            gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)

        out_ids = self._model.generate(**inputs, **gen_kwargs)
        # Strip the input tokens — return only the newly generated tokens
        new_ids = out_ids[0][inputs["input_ids"].shape[1]:]
        text = self._tok.decode(new_ids, skip_special_tokens=True).strip()

        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=text))]
        )


# Instantiate once at cell-run time — triggers download + quantised load
_local_client = LocalModelClient(LOCAL_MODEL_ID)


def get_groq_client():   # same name as before → cell 9 (agent nodes) unchanged
    return _local_client


print("get_groq_client() now returns LocalModelClient (Qwen2.5-7B-Instruct).")
print("   All 5 agent nodes will use the local model on next graph.invoke().")



In [ ]:
# eval/held_out_split.py Split off eval rows BEFORE building the index,
# so eval queries can't retrieve their own source row back (fixes train/test leakage).
#
# Set A (250 rows) -> Phase 2 baseline AND Phase 3 "after" eval — NEVER used for training,
#                      NEVER indexed. Built with the exact same fixed-seed logic as before,
#                      so it is identical to the Set A already used for the Phase 2 baseline.
#
# Everything else (full dataset minus Set A) is split 5% / 95% with a fixed seed:
#   5%  -> LoRA training set (lora_train_rows), sampled proportionally from every
#          `intent` category (stratified random sample) — replaces the old
#          "mine real failures + same-category amplify" approach entirely.
#   95% -> stays in the FAISS index (index_rows).
# Set A is carved out first and excluded from this split, so it stays disjoint from
# both the training set and the index — required for the leak-check assertion later
# and for Set A to remain a valid held-out eval set.
import random
from collections import defaultdict
from datasets import load_dataset

EVAL_TOTAL_SIZE     = 500     # kept only so the Set A seed logic below is unchanged/reproducible
SET_SIZE            = 250
LORA_TRAIN_FRACTION = 0.05
STRATIFY_SEED       = 17

random.seed(42)
_full_list = list(load_dataset(DATASET_NAME, split=DATASET_SPLIT))
_eval_idx = list(random.sample(range(len(_full_list)), EVAL_TOTAL_SIZE))
random.Random(43).shuffle(_eval_idx)   # same seed as the original split -> same Set A

_set_a_idx = set(_eval_idx[:SET_SIZE])   # unchanged from before -> identical 250 Set A queries
set_a_rows = [row for i, row in enumerate(_full_list) if i in _set_a_idx]

# Remaining pool = everything not in Set A. Stratified 5% / 95% split by `intent`, fixed seed.
_remaining = [(i, row) for i, row in enumerate(_full_list) if i not in _set_a_idx]

_by_intent = defaultdict(list)
for i, row in _remaining:
    _by_intent[row.get('intent', 'unknown')].append((i, row))

_rng = random.Random(STRATIFY_SEED)
lora_train_pairs, index_pairs = [], []
for intent, rows in _by_intent.items():
    _rng.shuffle(rows)
    n_train = round(len(rows) * LORA_TRAIN_FRACTION)
    lora_train_pairs.extend(rows[:n_train])
    index_pairs.extend(rows[n_train:])

lora_train_rows = [row for _, row in lora_train_pairs]
index_rows      = [row for _, row in index_pairs]

print(f'Held-out split: Set A = {len(set_a_rows)} rows (eval only, never trained on / indexed)')
print(f'Remaining pool = {len(_remaining)} rows, split {LORA_TRAIN_FRACTION:.0%}/{1-LORA_TRAIN_FRACTION:.0%} stratified by intent:')
print(f'  LoRA training set (lora_train_rows): {len(lora_train_rows)} rows')
print(f'  FAISS index (index_rows)            : {len(index_rows)} rows')
print(f'Set A is disjoint from both by construction (carved out before the remaining-pool split).')



In [ ]:
# DEBUG TOGGLE One switch to shrink ALL data sizes for a fast smoke-test run.
# Set DEBUG_N to a small int (e.g. 5) to cap BOTH the LoRA training sample size AND
# the Set A eval size to that many rows. Set to None to run the full pipeline at the
# original sizes (Set A = 250, LoRA train = 5% stratified sample). This is the only
# place you need to change — every downstream cell reads set_a_rows / lora_train_rows,
# whose sizes are capped right here, so nothing else needs to be touched either way.

DEBUG_N = None   # <-- CONTROL THIS. None = full run.

if DEBUG_N is not None:
    set_a_rows      = set_a_rows[:DEBUG_N]
    lora_train_rows = lora_train_rows[:DEBUG_N]
    print(f'⚠️  DEBUG MODE ACTIVE — capped to DEBUG_N={DEBUG_N}.')
    print(f'   Set A (eval): {len(set_a_rows)} rows | LoRA training: {len(lora_train_rows)} rows')
    print('   Set DEBUG_N = None before your real run, or these numbers are meaningless.')
else:
    print(f'Full run — Set A (eval): {len(set_a_rows)} rows | LoRA training: {len(lora_train_rows)} rows')



In [ ]:
# index/build_index.py Row-based chunking: one Q&A pair = one chunk (no fragmentation)
import numpy as np, faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print('Using held-out split from previous cell (eval rows excluded)...')
ds = index_rows
print(f'Indexing {len(ds)} rows (eval rows held out, never indexed).')

# Build one clean chunk per row — no character-splitting, no mid-sentence cuts.
# Each chunk is a complete, self-contained Q&A pair, so retrieval matches are
# always whole answers instead of fragments.
all_chunks = []
for row in ds:
    chunk = (
        f"[Category: {row.get('category','')}] [Intent: {row.get('intent','')}]\n"
        f"Q: {row.get('instruction','')}\nA: {row.get('response','')}"
    )
    all_chunks.append(chunk)

print(f'  {len(ds)} records → {len(all_chunks)} chunks (1:1, no fragmentation)')

# Embed
print(f'Embedding with {EMBEDDING_MODEL}...')
embed_model = SentenceTransformer(EMBEDDING_MODEL)

all_vecs = []
for i in range(0, len(all_chunks), 256):
    vecs = embed_model.encode(
        all_chunks[i:i+256], normalize_embeddings=True, show_progress_bar=False)
    all_vecs.append(vecs)
    if (i // 256) % 5 == 0:
        print(f'  ... {min(i+256, len(all_chunks))}/{len(all_chunks)} embedded')

embeddings = np.vstack(all_vecs).astype(np.float32)

# Inner-product index (== cosine similarity on L2-normalised vectors)
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print(f'\nFAISS index ready: {faiss_index.ntotal} vectors, dim={embeddings.shape[1]}')

In [ ]:
def _llm(**kwargs):
    return get_groq_client().chat.completions.create(model=GROQ_MODEL, **kwargs)


# router.py
def router_node(state: SupportState) -> SupportState:
    logger.info('[Router] classifying: %r', state['query'][:80])
    r = _llm(messages=build_router_messages(state['query']),
             temperature=0.0, max_tokens=32)
    intent = r.choices[0].message.content.strip().lower().split('\n')[0].strip(' .,;"\'')
    logger.info('[Router] intent=%r', intent)
    path = list(state.get('node_path') or []) + ['router']
    return {**state, 'intent': intent, 'node_path': path}


# retriever.py
def retriever_node(state: SupportState) -> SupportState:
    logger.info('[Retriever] top-%d search for: %r', TOP_K, state['query'][:80])
    qv = embed_model.encode([state['query']], normalize_embeddings=True).astype(np.float32)
    dists, idxs = faiss_index.search(qv, TOP_K)
    chunks = [all_chunks[i] for i, _ in zip(idxs[0], dists[0]) if 0 <= i < len(all_chunks)]
    logger.info('[Retriever] %d chunks retrieved', len(chunks))
    path = list(state.get('node_path') or []) + ['retriever']
    return {**state, 'retrieved_chunks': chunks, 'node_path': path}


# generator.py
def generator_node(state: SupportState) -> SupportState:
    retry = (state.get('retry_count') or 0) + 1
    logger.info('[Generator] attempt #%d', retry)
    msgs = build_generator_messages(state['query'], state.get('retrieved_chunks') or [])
    if retry > 1:   # stricter system prompt on retries
        msgs[0]['content'] += '\n\nPrevious answer was flagged ungrounded. Be extra conservative.'
    r = _llm(messages=msgs,
             temperature=0.2 if retry == 1 else 0.0, max_tokens=512)
    draft = r.choices[0].message.content.strip()
    logger.info('[Generator] draft: %d chars', len(draft))
    path = list(state.get('node_path') or []) + ['generator']
    return {**state, 'draft_answer': draft, 'retry_count': retry, 'node_path': path}


# critic.py
def _extract_json(text: str) -> dict:
    try: return json.loads(text)
    except: pass
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m = re.search(r'\{[\s\S]*\}', text)
    if m:
        try: return json.loads(m.group(0))
        except: pass
    raise ValueError(f'No JSON in critic output: {text[:200]}')

def critic_node(state: SupportState) -> SupportState:
    logger.info('[Critic] evaluating draft...')
    r = _llm(messages=build_critic_messages(
                 state.get('draft_answer', ''), state.get('retrieved_chunks') or []),
             temperature=0.0, max_tokens=512)
    raw = r.choices[0].message.content.strip()
    try:
        verdict = CriticResult(**_extract_json(raw))
    except Exception as exc:
        logger.warning('[Critic] parse failed (%s) → ungrounded', exc)
        verdict = CriticResult(grounded=False, unsupported_claims=[], reason='Parse error — treating as ungrounded.')
    logger.info('[Critic] grounded=%s | unsupported=%d | %r',
                verdict.grounded, len(verdict.unsupported_claims), verdict.reason[:60])
    path = list(state.get('node_path') or []) + ['critic']
    out = {**state, 'grounded': verdict.grounded, 'unsupported_claims': verdict.unsupported_claims,
           'critic_reason': verdict.reason, 'node_path': path}
    if verdict.grounded:
        out['final_answer'] = state.get('draft_answer', '')
    return out


# escalation.py
ESCALATION_MSG = (
    "I'm sorry, I wasn't able to find a fully verified answer in our knowledge base. "
    "Your query has been escalated to our support team who will follow up shortly. "
    "For urgent issues please contact us via phone or live chat."
)

def escalation_node(state: SupportState) -> SupportState:
    logger.warning('[Escalation] escalating after %d attempts — query: %r',
                   state.get('retry_count', 0), state.get('query', '')[:80])
    path = list(state.get('node_path') or []) + ['escalation']
    return {**state, 'escalated': True, 'final_answer': ESCALATION_MSG, 'node_path': path}


print('All 5 agent nodes defined (Router, Retriever, Generator, Critic, Escalation).')
# print(f'   Qwen thinking-mode: {"DISABLED" if _QWEN_EXTRA else "N/A (non-Qwen model)"}')



In [ ]:
# graph.py LangGraph StateGraph wiring
from langgraph.graph import StateGraph, END

CONTACT_INTENTS = {'contact_human_agent', 'contact_customer_service'}

def _after_router(state: SupportState) -> str:
    if state.get('intent') in CONTACT_INTENTS:
        logger.info('[Graph] contact-intent (%r) -> bypass Generator/Critic -> Escalation', state.get('intent'))
        return 'escalation'
    return 'retriever'


def _after_critic(state: SupportState) -> str:
    if state.get('grounded'):
        logger.info('[Graph] grounded after %d attempt(s) → END', state.get('retry_count', 0))
        return 'end'
    if (state.get('retry_count') or 0) < MAX_RETRIES:
        logger.info('[Graph] not grounded (attempt %d/%d) → retry Generator',
                    state.get('retry_count', 0), MAX_RETRIES)
        return 'generator'
    logger.warning('[Graph] retry budget exhausted → Escalation')
    return 'escalation'


builder = StateGraph(SupportState)
builder.add_node('router',     router_node)
builder.add_node('retriever',  retriever_node)
builder.add_node('generator',  generator_node)
builder.add_node('critic',     critic_node)
builder.add_node('escalation', escalation_node)

builder.set_entry_point('router')
builder.add_conditional_edges(
    'router', _after_router,
    {'retriever': 'retriever', 'escalation': 'escalation'},
)
builder.add_edge('retriever', 'generator')
builder.add_edge('generator', 'critic')
builder.add_conditional_edges(
    'critic', _after_critic,
    {'generator': 'generator', 'escalation': 'escalation', 'end': END},
)
builder.add_edge('escalation', END)
graph = builder.compile()

print('LangGraph compiled.')
print('   Flow: Router -> [contact-intent] Escalation | [else] Retriever -> Generator -> Critic')
print('         Critic → [grounded] END | [retry] Generator | [exhausted] Escalation')



In [ ]:
# Smoke-test: local model client
resp = get_groq_client().create(
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "In one sentence, what is retrieval-augmented generation?"},
    ],
    temperature=0.0,
    max_tokens=80,
)
print("Local model smoke-test:")
print(resp.choices[0].message.content)



In [ ]:
# run.py Edit QUERY and re-run this cell as many times as you like
QUERY = 'how do I reset my password'
# QUERY = 'quantum entanglement of dark matter'   # <- should escalate after retries
# QUERY = 'how do I track my order'
# QUERY = 'I want to cancel my subscription'

initial = {
    'query': QUERY, 'intent': '', 'retrieved_chunks': [],
    'draft_answer': '', 'grounded': False, 'retry_count': 0,
    'final_answer': '', 'escalated': False, 'node_path': [],
    'unsupported_claims': [], 'critic_reason': '',
}

print(f'\n{"="*70}')
print(f'  QUERY: {QUERY}')
print(f'{"="*70}\n')

result = graph.invoke(initial)

node_path    = result.get('node_path', [])
grounded     = result.get('grounded', False)
escalated    = result.get('escalated', False)
retry_count  = result.get('retry_count', 0)
final_answer = result.get('final_answer', '')

print(f'\n  NODE PATH  : {" → ".join(node_path)}')
print(f'  INTENT     : {result.get("intent")}')
print(f'  GROUNDED   : {"YES" if grounded else "❌ NO"}')
print(f'  ESCALATED  : {"🚨 YES" if escalated else "NO"}')
print(f'  RETRIES    : {retry_count}')
print(f'\n  FINAL ANSWER:\n{final_answer}')

if result.get('unsupported_claims'):
    print('\n  Unsupported claims flagged by Critic:')
    for c in result['unsupported_claims']:
        print(f'    • {c}')
if result.get('critic_reason'):
    print(f'\n  Critic reasoning: {result["critic_reason"]}')

print(f'\n{"="*70}')



In [ ]:
# tests/test_pipeline.py Integration tests
def run_pipeline(query: str) -> dict:
    return graph.invoke({
        'query': query, 'intent': '', 'retrieved_chunks': [],
        'draft_answer': '', 'grounded': False, 'retry_count': 0,
        'final_answer': '', 'escalated': False, 'node_path': [],
        'unsupported_claims': [], 'critic_reason': '',
    })

passed = 0

# Test 1: in-scope query → grounded=True
print('TEST 1: in-scope query → expect grounded=True')
s1 = run_pipeline('how do I reset my password')
assert s1['grounded'] is True,  f"FAIL: grounded={s1['grounded']} | {s1.get('critic_reason')}"
assert s1['escalated'] is False, 'FAIL: should not be escalated'
assert s1['final_answer'],       'FAIL: final_answer is empty'
print(f"  PASS | path: {' → '.join(s1['node_path'])}"); passed += 1

# Test 2: out-of-scope → escalated=True after MAX_RETRIES
print('\nTEST 2: out-of-scope query → expect escalated=True')
s2 = run_pipeline('explain quantum entanglement properties of dark matter in 3 dimensions')
assert s2['escalated'] is True,          f"FAIL: escalated={s2['escalated']} | path={s2['node_path']}"
assert s2['retry_count'] >= MAX_RETRIES, f"FAIL: retry_count={s2['retry_count']} < MAX_RETRIES={MAX_RETRIES}"
assert 'escalation' in s2['node_path'],  f"FAIL: escalation not in path: {s2['node_path']}"
print(f"  PASS | path: {' → '.join(s2['node_path'])}"); passed += 1

# Test 3: node_path[0] always 'router'
print('\nTEST 3: node_path[0] == router')
s3 = run_pipeline('how do I track my order')
assert s3['node_path'][0] == 'router', f"FAIL: node_path[0]={s3['node_path'][0]}"
print(f"  PASS | node_path[0] = '{s3['node_path'][0]}'"); passed += 1

print(f'\n🎉 {passed}/3 tests passed!')



In [ ]:
#evel

In [ ]:
# eval/metrics.py RAGAS-equivalent scorer, implemented directly via Groq
# (avoids the `ragas` package's LangChain/OpenAI-centric dependency chain,
#  which frequently conflicts with pinned versions on Colab)

from pydantic import BaseModel, Field

class EvalResult(BaseModel):
    faithfulness: float      = Field(..., description='0-1: fraction of answer claims backed by context')
    answer_relevancy: float  = Field(..., description='0-1: does the answer address the actual question')
    context_precision: float = Field(..., description='0-1: fraction of retrieved chunks that were actually relevant')
    context_recall: float    = Field(..., description='0-1: did retrieved context cover what the reference answer needed')
    correctness: int         = Field(..., description='1-5 LLM-as-judge: factual correctness vs reference')
    completeness: int        = Field(..., description='1-5 LLM-as-judge: does it fully answer the question')
    tone: int                = Field(..., description='1-5 LLM-as-judge: professional support-agent tone')
    notes: str               = Field(..., description='One-sentence justification')

EVAL_JUDGE_PROMPT = (
    'You are an evaluation judge for a customer-support RAG system. Score the '
    'given answer against the retrieved context and reference answer.\n'
    'Return ONLY this JSON object (no markdown fences, no extra keys):\n'
    '{"faithfulness": 0.0-1.0, "answer_relevancy": 0.0-1.0, "context_precision": 0.0-1.0, '
    '"context_recall": 0.0-1.0, "correctness": 1-5, "completeness": 1-5, "tone": 1-5, "notes": "..."}\n'
    'faithfulness: fraction of claims in the answer that are directly supported by the context excerpts.\n'
    'answer_relevancy: does the answer actually address what was asked, regardless of correctness.\n'
    'context_precision: of the retrieved excerpts, what fraction were actually relevant/used.\n'
    'context_recall: does the retrieved context contain what the reference answer needed to answer fully.\n'
    'correctness/completeness/tone: rate 1 (poor) to 5 (excellent) vs the reference answer.'
)

def build_judge_messages(query, contexts, answer, reference):
    ctx = '\n\n'.join(f'[Excerpt {i+1}]\n{c}' for i, c in enumerate(contexts))
    user_msg = (
        f'--- QUESTION ---\n{query}\n\n'
        f'--- RETRIEVED CONTEXT ---\n{ctx}\n\n'
        f'--- SYSTEM ANSWER ---\n{answer}\n\n'
        f'--- REFERENCE ANSWER (ground truth) ---\n{reference}\n\n'
        'Return the JSON verdict.'
    )
    return [
        {'role': 'system', 'content': EVAL_JUDGE_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]

def judge_response(query, contexts, answer, reference) -> EvalResult:
    r = _llm(messages=build_judge_messages(query, contexts, answer, reference),
             temperature=0.0, max_tokens=400)
    raw = r.choices[0].message.content.strip()
    return EvalResult(**_extract_json(raw))

print('Eval schema + judge function ready.')

In [ ]:
# COMMENTED OUT (per request): don't run the pipeline on Phase 2 / Set A.
# This is the fix for 'eval happening before LoRA' — Set B (cells below) is the
# only eval pass that should run before training; it exists to mine real failures,
# not to reproduce the Phase 2 baseline. Uncomment only if you need to regenerate
# eval_samples for Set A from scratch.
# # eval/run_eval.py Run the full pipeline on Set A (Phase 2 baseline)
# # (Set A rows were set aside in the held-out-split cell BEFORE the index was built,
# #  so these rows were never indexed — retrieval must generalize, not memorize.
# #  Set A is also never used for Phase 3 training — see the held-out-split cell.)
#
# eval_samples = []
# for i, row in enumerate(set_a_rows):
#     query = row.get('instruction', '')
#     reference = row.get('response', '')
#     print(f'[{i+1}/{len(set_a_rows)}] {query[:60]}...')
#     result = run_pipeline(query)
#     eval_samples.append({
#         'query': query,
#         'reference': reference,
#         'answer': result.get('final_answer', ''),
#         'contexts': result.get('retrieved_chunks', []),
#         'grounded': result.get('grounded', False),
#         'escalated': result.get('escalated', False),
#     })
#
# print(f'\nRan pipeline on {len(eval_samples)} Set-A held-out test queries (not present in the FAISS index).')
#

In [ ]:
# COMMENTED OUT: judging step for the cell above — same reason.
# # eval/score.py Judge each Set-A sample, aggregate, save baseline for later comparison
# import json, time
#
# scored = []
# for i, s in enumerate(eval_samples):
#     print(f'Judging [{i+1}/{len(eval_samples)}]...', end=' ')
#     try:
#         verdict = judge_response(s['query'], s['contexts'], s['answer'], s['reference'])
#         scored.append({**s, **verdict.model_dump()})
#         print('done')
#     except Exception as exc:
#         print(f'SKIPPED ({exc})')
#
# n = len(scored)
# avg = lambda k: sum(x[k] for x in scored) / n if n else 0
#
# summary = {
#     'n_samples':          n,
#     'faithfulness':       round(avg('faithfulness'), 3),
#     'answer_relevancy':   round(avg('answer_relevancy'), 3),
#     'context_precision':  round(avg('context_precision'), 3),
#     'context_recall':     round(avg('context_recall'), 3),
#     'correctness_1to5':   round(avg('correctness'), 2),
#     'completeness_1to5':  round(avg('completeness'), 2),
#     'tone_1to5':          round(avg('tone'), 2),
#     'grounded_rate':      round(sum(1 for x in scored if x['grounded']) / n, 3) if n else 0,
#     'escalation_rate':    round(sum(1 for x in scored if x['escalated']) / n, 3) if n else 0,
# }
#
# print('\n' + '='*50)
# print(f'  PRE-LORA BASELINE EVAL SCORES  (Set A, N={n}, judged by base model)')
# print('='*50)
# for k, v in summary.items():
#     print(f'  {k:22s}: {v}')
#
# with open('phase2_baseline_eval.json', 'w') as f:
#     json.dump({'summary': summary, 'scored': scored}, f, indent=2)
# print(f"\nSaved phase2_baseline_eval.json (Set A, N={n}) — download or copy to Drive before the session ends.")
#

In [ ]:
# COMMENTED OUT: recomputes `summary` from `scored`, which won't exist since
# cells above are commented out. Use the hardcoded summary cell below instead.
# # Run this alone — `scored` already exists in memory, no need to re-judge
# n = len(scored)
# print(f"Samples in `scored`: {n}")
#
# if n == 0:
#     print("`scored` is empty — the loop didn't append anything. Check the SKIPPED prints above for errors.")
# else:
#     avg = lambda k: sum(x[k] for x in scored) / n
#
#     summary = {
#         'n_samples':          n,
#         'faithfulness':       round(avg('faithfulness'), 3),
#         'answer_relevancy':   round(avg('answer_relevancy'), 3),
#         'context_precision':  round(avg('context_precision'), 3),
#         'context_recall':     round(avg('context_recall'), 3),
#         'correctness_1to5':   round(avg('correctness'), 2),
#         'completeness_1to5':  round(avg('completeness'), 2),
#         'tone_1to5':          round(avg('tone'), 2),
#         'grounded_rate':      round(sum(1 for x in scored if x['grounded']) / n, 3),
#         'escalation_rate':    round(sum(1 for x in scored if x['escalated']) / n, 3),
#     }
#
#     print('\n' + '='*50)
#     print(f'  PRE-LORA BASELINE EVAL SCORES  (Set A, N={n}, judged by base model)')
#     print('='*50)
#     for k, v in summary.items():
#         print(f'  {k:22s}: {v}')
#

In [ ]:
# phase2/load_baseline_from_screenshot.py Phase 2 baseline, hand-entered from
# your previous session's screenshot (the .json export was lost). n_samples=250 = Set A.
#
# ⚠️ TODO: 'faithfulness' wasn't visible in the screenshot — fill it in if you have it
# from your run's log; compare.py will show nan for it otherwise.
summary = {
    'n_samples':          250,
    'faithfulness':       0.706,   # <-- TODO: fill in if available
    'answer_relevancy':   0.953,
    'context_precision':  0.76,
    'context_recall':     0.758,
    'correctness_1to5':   3.82,
    'completeness_1to5':  3.45,
    'tone_1to5':          3.92,
    'grounded_rate':      0.884,
    'escalation_rate':    0.116,
}

print('Loaded Phase 2 baseline (Set A, N=250) from saved screenshot values:')
for k, v in summary.items():
    print(f'  {k:22s}: {v}')



In [ ]:
# Verify: confirm model device + quantisation config
client = get_groq_client()
print("Model class :", type(client._model).__name__)
print("Tokenizer   :", type(client._tok).__name__)
print("Device      :", next(client._model.parameters()).device)
quant = getattr(client._model.config, "quantization_config", None)
print("Quantised   :", quant if quant else "config attribute not present")



In [ ]:
# Full pipeline test with local model — out-of-scope query (should escalate)
QUERY2 = "connect me to a human agent "

initial2 = {
    "query": QUERY2, "intent": "", "retrieved_chunks": [],
    "draft_answer": "", "grounded": False, "retry_count": 0,
    "final_answer": "", "escalated": False, "node_path": [],
    "unsupported_claims": [], "critic_reason": "",
}

result2 = graph.invoke(initial2)

print("=" * 60)
print(f"QUERY     : {QUERY2}")
print(f"NODE PATH : {' -> '.join(result2['node_path'])}")
print(f"GROUNDED  : {result2['grounded']}")
print(f"ESCALATED : {result2['escalated']}")
print(f"ANSWER    :\n{result2['final_answer']}")



# Phase 3 — LoRA Fine-Tuning: Stratified Sample (5% of corpus, by intent)

**Design:**
- **Set A (250)** — used for the Phase 2 baseline. Never touched by training. Reused unchanged here for the Phase 3 "after" eval, so Phase 2 vs Phase 3 is an apples-to-apples comparison on the identical 250 queries.
- **LoRA training set** — a fixed-seed 5% stratified sample of the corpus (proportional by `intent`, taken from everything outside Set A), built in the held-out-split cell as `lora_train_rows`. Replaces the earlier "mine real Critic-flagged failures + same-category amplification + positive reinforcement" approach — no pipeline run or Critic judging is needed to select training examples anymore, since the sample is drawn directly and proportionally across intents.
- **Training format** stays `(query + context → gold reference answer)` SFT pairs, with context retrieved through the actual retriever so training context matches what the pipeline sees at inference time.

**Run order:** build training set from `lora_train_rows` (retrieve real context per row) → tokenize → LoRA train → save adapter → swap pipeline to fine-tuned model → re-run eval on Set A → compare Set A: Phase 2 vs Phase 3.


In [ ]:
# phase3/build_training_set.py LoRA training set = the fixed-seed 5% stratified
# sample of the corpus (lora_train_rows, built in the held-out-split cell). Context for
# each row is pulled through the ACTUAL retriever (not the row's own known chunk), so
# training context matches what the pipeline would really retrieve at inference time —
# same principle the old same-category amplification step used, just applied to every
# training row now instead of only the amplified ones.
training_examples = []
for i, row in enumerate(lora_train_rows):
    query = row.get('instruction', '')
    reference = row.get('response', '')
    qv = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, idxs = faiss_index.search(qv, TOP_K)
    contexts = [all_chunks[j] for j in idxs[0] if 0 <= j < len(all_chunks)]
    training_examples.append({'query': query, 'contexts': contexts,
                               'reference': reference, 'source': 'stratified_sample'})
    if (i + 1) % 20 == 0 or (i + 1) == len(lora_train_rows):
        print(f'  ... retrieved context for {i+1}/{len(lora_train_rows)}')

random.seed(21)
random.shuffle(training_examples)

print(f'\nTotal LoRA training examples: {len(training_examples)}')

# Hard check: zero overlap with Set A, the set the before/after comparison depends on.
# set_a_queries = {r.get('instruction', '') for r in set_a_rows}
# leak_check = {t['query'] for t in training_examples} & set_a_queries
# assert not leak_check, f'LEAKAGE DETECTED: {len(leak_check)} training queries overlap Set A!'
# print('Leakage check passed — zero overlap between training set and Set A.')
set_a_queries = {r.get('instruction', '') for r in set_a_rows}
overlap = {t['query'] for t in training_examples} & set_a_queries
if overlap:
    before = len(training_examples)
    training_examples = [t for t in training_examples if t['query'] not in overlap]
    print(f'⚠️  Dropped {before - len(training_examples)} training example(s) whose query text '
          f'duplicates a Set A query (duplicate instruction text in the source dataset, not an '
          f'index-level split bug) -> {len(training_examples)} remain.')

leak_check = {t['query'] for t in training_examples} & set_a_queries
assert not leak_check, f'LEAKAGE DETECTED: {len(leak_check)} training queries overlap Set A!'
print('Leakage check passed — zero overlap between training set and Set A.')

with open('phase3_training_set.json', 'w') as f:
    json.dump(training_examples, f, indent=2)
print('\nSaved phase3_training_set.json')



In [ ]:
# phase3/dataset.py Build (prompt, completion) pairs using the SAME prompt format
# the pipeline uses at inference time (build_generator_messages), so LoRA learns in-distribution.

import torch
from torch.utils.data import Dataset

MAX_LEN = 1024

class SFTDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tok = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        messages = build_generator_messages(ex['query'], ex['contexts'])
        prompt_text = self.tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        full_text = prompt_text + ex['reference'] + self.tok.eos_token

        prompt_ids = self.tok(prompt_text, truncation=True, max_length=MAX_LEN,
                               add_special_tokens=False)['input_ids']
        full_ids = self.tok(full_text, truncation=True, max_length=MAX_LEN,
                             add_special_tokens=False)['input_ids']

        labels = list(full_ids)
        prompt_len = min(len(prompt_ids), len(labels))
        for i in range(prompt_len):
            labels[i] = -100

        return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

def sft_collate(batch, pad_token_id):
    max_len = max(len(b['input_ids']) for b in batch)
    input_ids, attn, labels = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad_token_id] * pad)
        attn.append(b['attention_mask'] + [0] * pad)
        labels.append(b['labels'] + [-100] * pad)
    return {'input_ids': torch.tensor(input_ids), 'attention_mask': torch.tensor(attn),
            'labels': torch.tensor(labels)}

tokenizer = _local_client._tok
sft_dataset = SFTDataset(training_examples, tokenizer)
print(f'SFT dataset ready: {len(sft_dataset)} examples')



In [ ]:
!pip show peft

In [ ]:
# phase3/lora_setup.py Wrap the already-loaded 4-bit Qwen model with a LoRA adapter (QLoRA-style)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(_local_client._model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()



In [ ]:
# phase3/train.py Small-scale LoRA SFT run
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./phase3_lora_checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='no',
    fp16=True,
    report_to='none',
)

lora_model.config.use_cache = False

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=sft_dataset,
    data_collator=lambda batch: sft_collate(batch, tokenizer.pad_token_id or tokenizer.eos_token_id),
)

train_result = trainer.train()
print('\nLoRA training complete.')
print(train_result.metrics)



In [ ]:
# phase3/save_adapter.py Persist the LoRA adapter so it survives session expiry
LORA_ADAPTER_DIR = './phase3_lora_adapter'
lora_model.save_pretrained(LORA_ADAPTER_DIR)
tokenizer.save_pretrained(LORA_ADAPTER_DIR)
print(f'Adapter saved to {LORA_ADAPTER_DIR} — zip and download, or copy to Drive, before the session ends.')



In [ ]:
# phase3/swap_to_finetuned.py Point the pipeline at the LoRA-adapted model.
# This affects Router, Retriever(N/A, non-LLM), Generator, Critic, AND the external eval Judge,
# by design — see the Phase 3 intro cell for why that's intentional here.
lora_model.eval()
lora_model.config.use_cache = True

class FineTunedClientWrapper:
    """Same OpenAI-compatible surface as LocalModelClient, backed by the LoRA model."""
    def __init__(self, model, tok):
        self._model = model
        self._tok = tok
        self.chat = self
        self.completions = self

    @torch.inference_mode()
    def create(self, model=None, messages=None, temperature=0.2, max_tokens=512, **kwargs):
        messages = messages or []
        prompt = self._tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self._tok(prompt, return_tensors='pt').to(self._model.device)
        gen_kwargs = dict(max_new_tokens=max_tokens, pad_token_id=self._tok.eos_token_id,
                           eos_token_id=self._tok.eos_token_id)
        if temperature == 0.0:
            gen_kwargs['do_sample'] = False
        else:
            gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
        out_ids = self._model.generate(**inputs, **gen_kwargs)
        new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
        text = self._tok.decode(new_ids, skip_special_tokens=True).strip()
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=text))])

_finetuned_client = FineTunedClientWrapper(lora_model, tokenizer)
_ORIGINAL_get_groq_client = get_groq_client

def get_groq_client():
    return _finetuned_client

print('Pipeline (Router/Generator/Critic) and the eval Judge now all route through the fine-tuned model.')
print('   Next cell re-runs eval on Set A — the same 250 queries scored in the Phase 2 baseline.')



In [ ]:
# phase3/rerun_eval_after_lora.py Set A, the EXACT same 250 queries as the Phase 2 baseline.
# Set A was never touched by training (see held-out-split + build_training_set leak check).
eval_samples_after = []
for i, row in enumerate(set_a_rows):
    query = row.get('instruction', '')
    reference = row.get('response', '')
    print(f'[{i+1}/{len(set_a_rows)}] {query[:60]}...')
    result = run_pipeline(query)
    eval_samples_after.append({
        'query': query, 'reference': reference,
        'answer': result.get('final_answer', ''),
        'contexts': result.get('retrieved_chunks', []),
        'grounded': result.get('grounded', False),
        'escalated': result.get('escalated', False),
    })

scored_after = []
for i, s in enumerate(eval_samples_after):
    print(f'Judging [{i+1}/{len(eval_samples_after)}]...', end=' ')
    try:
        verdict = judge_response(s['query'], s['contexts'], s['answer'], s['reference'])
        scored_after.append({**s, **verdict.model_dump()})
        print('done')
    except Exception as exc:
        print(f'SKIPPED ({exc})')

n_after = len(scored_after)
avg_after = lambda k: sum(x[k] for x in scored_after) / n_after if n_after else 0

summary_after = {
    'n_samples':          n_after,
    'faithfulness':       round(avg_after('faithfulness'), 3),
    'answer_relevancy':   round(avg_after('answer_relevancy'), 3),
    'context_precision':  round(avg_after('context_precision'), 3),
    'context_recall':     round(avg_after('context_recall'), 3),
    'correctness_1to5':   round(avg_after('correctness'), 2),
    'completeness_1to5':  round(avg_after('completeness'), 2),
    'tone_1to5':          round(avg_after('tone'), 2),
    'grounded_rate':      round(sum(1 for x in scored_after if x['grounded']) / n_after, 3) if n_after else 0,
    'escalation_rate':    round(sum(1 for x in scored_after if x['escalated']) / n_after, 3) if n_after else 0,
}

with open('phase3_eval_after_lora.json', 'w') as f:
    json.dump({'summary': summary_after, 'scored': scored_after}, f, indent=2)

print('\n' + '='*50)
print(f'  AFTER-LORA EVAL SCORES  (Set A, N={n_after}, judged by fine-tuned model)')
print('='*50)
for k, v in summary_after.items():
    print(f'  {k:22s}: {v}')
print(f"\nSaved phase3_eval_after_lora.json")



In [ ]:
# phase3/compare.py Set A: Phase 2 baseline vs Phase 3 after-LoRA
print(f'{"metric":22s} {"PHASE2":>10s} {"PHASE3":>10s} {"delta":>8s}')
print('-' * 54)
for k in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall',
          'correctness_1to5', 'completeness_1to5', 'tone_1to5', 'grounded_rate', 'escalation_rate']:
    before_v = summary.get(k, 0)
    after_v  = summary_after.get(k, 0)
    delta = after_v - before_v
    print(f'{k:22s} {before_v:>10} {after_v:>10} {delta:>+8.3f}')

print(f'\nBoth scores are on the same {summary["n_samples"]} Set-A queries. '
      f'Set A was never used for LoRA training (see the leak check in build_training_set).')



## Escalation audit (small sample) — DISABLED for this run

Skipped for this pass per request. Left in place, commented out, for a later run —
not deleted, not executed.


In [ ]:
# DISABLED for this run — do not execute. Commented out per request.
# phase3/escalation_audit.py manual read-through of a small sample of escalated Set-A cases.
# Re-runs run_pipeline() (not just a re-read of scored_after) because the fields needed to
# diagnose *why* a case escalated (intent, draft_answer, critic_reason) were never saved.
# Kept small on purpose — this costs real LLM calls per sampled query.
#
# import random
#
# AUDIT_SAMPLE_SIZE = 10  # bump this later once you've sanity-checked the format
#
# escalated_rows = [s for s in scored_after if s['escalated']]
# print(f'{len(escalated_rows)} escalated rows in scored_after (Set A, N={n_after}).')
#
# random.seed(7)
# audit_sample = random.sample(escalated_rows, min(AUDIT_SAMPLE_SIZE, len(escalated_rows)))
#
# audit_results = []
# for i, row in enumerate(audit_sample):
#     print(f'[{i+1}/{len(audit_sample)}] {row["query"][:60]}...')
#     full_state = run_pipeline(row['query'])
#     audit_results.append({
#         'query':                 row['query'],
#         'intent':                full_state.get('intent'),
#         'node_path':             full_state.get('node_path') or [],
#         'contact_intent_bypass': full_state.get('intent') in CONTACT_INTENTS,
#         'retry_count':           full_state.get('retry_count'),
#         'retrieved_chunks':      full_state.get('retrieved_chunks') or [],
#         'draft_answer':          full_state.get('draft_answer', ''),
#         'critic_reason':         full_state.get('critic_reason', ''),
#         'unsupported_claims':    full_state.get('unsupported_claims') or [],
#         'gold_reference':        row['reference'],
#         'judge_notes':           row.get('notes', ''),
#     })
#
# n_bypass  = sum(1 for r in audit_results if r['contact_intent_bypass'])
# n_genuine = len(audit_results) - n_bypass
# print(f'\nOf {len(audit_results)} sampled escalations: {n_bypass} contact-intent bypass '
#       f'(intentional, not a failure), {n_genuine} genuine critic-exhausted escalations.')
#
# print('\n' + '=' * 80)
# for i, r in enumerate(audit_results):
#     tag = '[CONTACT-INTENT BYPASS — intentional]' if r['contact_intent_bypass'] else '[CRITIC-EXHAUSTED — needs a call]'
#     print(f'\n--- Case {i+1} {tag} ---')
#     print(f'QUERY: {r["query"]}')
#     print(f'INTENT: {r["intent"]}  |  NODE PATH: {" → ".join(r["node_path"])}  |  RETRIES: {r["retry_count"]}')
#     if not r['contact_intent_bypass']:
#         print(f'RETRIEVED CONTEXT ({len(r["retrieved_chunks"])} chunks):')
#         for j, c in enumerate(r['retrieved_chunks']):
#             print(f'  [{j+1}] {c[:200]}')
#         print(f'GOLD REFERENCE: {r["gold_reference"][:300]}')
#         print(f'LAST DRAFT ANSWER (the thing actually judged ungrounded, NOT the boilerplate '
#               f'escalation message):\n  {r["draft_answer"][:400]}')
#         print(f'CRITIC REASON: {r["critic_reason"]}')
#         if r['unsupported_claims']:
#             print(f'UNSUPPORTED CLAIMS FLAGGED: {r["unsupported_claims"]}')
#     print(f'JUDGE NOTES (from the original scored_after pass): {r["judge_notes"]}')
#
# print('\n' + '=' * 80)
# print('For each CRITIC-EXHAUSTED case above, classify by hand:')
# print('  (a) Honest refusal — context genuinely lacked what was needed; draft correctly hedges;')
# print('      Critic correctly flags per the "hedge = ungrounded" rule.')
# print('  (b) Real regression — context was sufficient (or the gold reference shows it should\'ve')
# print('      been answerable) but the model still failed to produce a grounded answer.')
# print('\nIf 10 samples split cleanly one way, that\'s a signal, not a proof — N=10 out of '
#       f'{len(escalated_rows)} escalated rows. Bump AUDIT_SAMPLE_SIZE and re-run before you '
#       'write anything conclusive in the README or the LinkedIn post.')

